In [1]:
import torch
import numpy as np


def set_seed(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
    return


set_seed(42)

In [97]:
def create_dummy(weights, num_samples, bias=False):
    # Generate random features X and target y based on linear relationship with weights
    X = np.random.randn(num_samples, len(weights))
    y = X @ weights
    if bias:
        y += np.random.randn(num_samples)
    return X, y


X, y = create_dummy(weights=[1, 1], num_samples=1000)
X.shape, y.shape

((1000, 2), (1000,))

In [2]:
X = np.linspace(-5, 5, 100).reshape(-1, 1)  # shape (100, 1)
y = 2 * X.squeeze()

In [3]:
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split

from tabpfn import TabPFNRegressor

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.5, random_state=42
)

# Initialize the regressor
regressor = TabPFNRegressor(device="cuda", n_estimators=1)
regressor.fit(X_train, y_train)

# Predict on the test set
# predictions = regressor.predict(X_test)

# # Evaluate the model
# mse = mean_squared_error(y_test, predictions)
# r2 = r2_score(y_test, predictions)

# print("Mean Squared Error (MSE):", mse)
# print("R² Score:", r2)

TabPFNRegressor(device='cuda', n_estimators=1)

In [4]:
model = regressor.model_
model

PerFeatureTransformer(
  (encoder): SequentialEncoder(
    (0): RemoveEmptyFeaturesEncoderStep()
    (1): NanHandlingEncoderStep()
    (2): VariableNumFeaturesEncoderStep()
    (3): InputNormalizationEncoderStep()
    (4): VariableNumFeaturesEncoderStep()
    (5): LinearInputEncoderStep(
      (layer): Linear(in_features=4, out_features=192, bias=False)
    )
  )
  (y_encoder): SequentialEncoder(
    (0): NanHandlingEncoderStep()
    (1): LinearInputEncoderStep(
      (layer): Linear(in_features=2, out_features=192, bias=True)
    )
  )
  (transformer_encoder): LayerStack(
    (layers): ModuleList(
      (0-11): 12 x PerFeatureEncoderLayer(
        (self_attn_between_features): MultiHeadAttention()
        (self_attn_between_items): MultiHeadAttention()
        (mlp): MLP(
          (linear1): Linear(in_features=192, out_features=768, bias=False)
          (linear2): Linear(in_features=768, out_features=192, bias=False)
        )
        (layer_norms): ModuleList(
          (0-2): 3 x 

In [5]:
import torch.nn as nn

activations = {}


def get_activation(name):
    def hook(model, input, output):
        # For Transformer blocks, 'output' is often a tuple.
        # We're usually interested in the first element (the hidden states).
        activations[name] = output[0].detach()
        print(name, activations[name].shape)

    return hook


hook_handles = []
for i, layer in enumerate(model.transformer_encoder.layers):
    handle = layer.register_forward_hook(get_activation(f"layer_{i}"))
    hook_handles.append(handle)

with torch.no_grad():
    prediction_probabilities = regressor.predict(X_test)

for handle in hook_handles:
    handle.remove()

layer_0 torch.Size([100, 3, 192])
layer_1 torch.Size([100, 3, 192])
layer_2 torch.Size([100, 3, 192])
layer_3 torch.Size([100, 3, 192])
layer_4 torch.Size([100, 3, 192])
layer_5 torch.Size([100, 3, 192])
layer_6 torch.Size([100, 3, 192])
layer_7 torch.Size([100, 3, 192])
layer_8 torch.Size([100, 3, 192])
layer_9 torch.Size([100, 3, 192])
layer_10 torch.Size([100, 3, 192])
layer_11 torch.Size([100, 3, 192])


In [95]:
device = "cuda" if torch.cuda.is_available() else "cpu"


# Define a simple linear probe
class LinearProbe(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(LinearProbe, self).__init__()
        self.linear = nn.Linear(input_dim, output_dim)

    def forward(self, x):
        return self.linear(x)


layer_name = "layer_11"

activation_size = activations[layer_name].shape[1] * activations[layer_name].shape[2]

activation_set = activations[layer_name].view(-1, activation_size)[501:]
target = torch.ones(activation_set.shape[0])
activation_train, activation_test, target_train, target_test = train_test_split(
    activation_set, target, test_size=0.5, random_state=42
)

# Move data to device and ensure consistent dtype
activation_train = activation_train.to(device).float()
activation_test = activation_test.to(device).float()
target_train = target_train.to(device).float()
target_test = target_test.to(device).float()

# Initialize the linear probe
linear_probe = LinearProbe(input_dim=activation_size, output_dim=1).to(device)

# Define the loss function and optimizer
criterion = nn.MSELoss()
optimizer = torch.optim.SGD(linear_probe.parameters(), lr=0.001)

# Training loop
num_epochs = 100

for epoch in range(num_epochs):
    optimizer.zero_grad()
    outputs = linear_probe(activation_train).squeeze()
    loss = criterion(outputs, target_train)
    loss.backward()
    optimizer.step()

# Evaluate the model
with torch.no_grad():
    test_outputs = linear_probe(activation_test).squeeze()
    test_loss = criterion(test_outputs, target_test)
    r2 = r2_score(target_test.cpu().numpy(), test_outputs.cpu().numpy())
    print(f"Test Loss: {test_loss.item():.4f}, R2 Score: {r2:.4f}")

Test Loss: 0.0209, R2 Score: 0.0000
